### The Run Expectancy Matrix


There are 24 possible states of a team's inning, based on:

- How many outs (3 possibilities)
- Runners on base (8 possibilites)

Base states are coded with four digits as follows:

- The first three digits corresponds to a base
- 0 means base empty, 1 means a man on base
- The fourth digit refers to the number of outs.

### Coding Run Expectancy

In [1]:
import pandas as pd
import numpy as np

In [2]:
MLBAM18 = pd.read_csv('Data/MLBAM18.csv')

MLBAM18.drop(['Unnamed: 0'], axis=1, inplace=True)

pd.set_option('display.max_columns', 100)
MLBAM18.head()

,inning,batterId,pitcherId,event,x,y,ab_num,timestamp,stand,throws,runnerMovement,half,balls,strikes,endOuts,actionId,description,game_type,home_team,home_teamId,home_lg,away_team,away_teamId,away_lg,venueId,stadium,field_teamId,playerId.1B,playerId.2B,playerId.3B,playerId.C,playerId.CF,playerId.LF,playerId.RF,playerId.SS,batterPos,batterName,pitcherName,runsOnPlay,startOuts,runsInInning,runsITD,runsFuture,start1B,start2B,start3B,end1B,end2B,end3B,outsInInning,startCode,endCode,fielderId,gameId,isPA,isAB,isHit,isBIP,our.x,our.y,r,theta
0,1,664023,570632,Home Run,233.22,70.48,1,2018-03-29 16:43:11,L,R,[664023:::T:Home Run],top,0,0,0,NaN,Ian Happ homers (1) on a fly ball to right fie...,R,mia,146,NL,chn,112,NL,4169,Marlins Park,146,571506,516770,605119,595453,621446,518618,643265,500743,CF,"Happ, I",Urena,1,0,3,0,3,NaN,NaN,NaN,NaN,NaN,NaN,3,0,0,NaN,gid_2018_03_29_chnmlb_miamlb_1,True,True,True,False,270.081515,320.743636,419.309557,0.870937
1,1,592178,570632,Walk,NaN,NaN,2,2018-03-29 16:43:56,R,R,[592178::1B::Walk],top,4,2,0,NaN,Kris Bryant walks.,R,mia,146,NL,chn,112,NL,4169,Marlins Park,146,571506,516770,605119,595453,621446,518618,643265,500743,3B,Bryant,Urena,0,0,3,1,2,NaN,NaN,NaN,592178.0,NaN,NaN,3,0,1,NaN,gid_2018_03_29_chnmlb_miamlb_1,True,False,False,False,NaN,NaN,NaN,NaN
2,1,519203,570632,Hit By Pitch,NaN,NaN,3,2018-03-29 16:46:24,L,R,[592178:1B:2B::Hit By Pitch][519203::1B::Hit B...,top,1,2,0,NaN,Anthony Rizzo hit by pitch. Kris Bryant to ...,R,mia,146,NL,chn,112,NL,4169,Marlins Park,146,571506,516770,605119,595453,621446,518618,643265,500743,1B,Rizzo,Urena,0,0,3,1,2,592178.0,NaN,NaN,519203.0,592178.0,NaN,3,1,3,NaN,gid_2018_03_29_chnmlb_miamlb_1,True,False,False,False,NaN,NaN,NaN,NaN
3,1,575929,570632,Strikeout,NaN,NaN,4,2018-03-29 16:48:44,R,R,NaN,top,2,3,1,NaN,Willson Contreras strikes out swinging.,R,mia,146,NL,chn,112,NL,4169,Marlins Park,146,571506,516770,605119,595453,621446,518618,643265,500743,C,Contreras,Urena,0,0,3,1,2,519203.0,592178.0,NaN,519203.0,592178.0,NaN,3,3,3,NaN,gid_2018_03_29_chnmlb_miamlb_1,True,True,False,False,NaN,NaN,NaN,NaN
4,1,656941,570632,Groundout,148.05,164.76,5,2018-03-29 16:52:10,L,R,[519203:1B:2B::Groundout][592178:2B:3B::Ground...,top,2,2,2,NaN,"Kyle Schwarber grounds out, first baseman Just...",R,mia,146,NL,chn,112,NL,4169,Marlins Park,146,571506,516770,605119,595453,621446,518618,643265,500743,LF,Schwarber,Urena,0,1,3,1,2,519203.0,592178.0,NaN,NaN,519203.0,592178.0,3,3,6,571506.0,gid_2018_03_29_chnmlb_miamlb_1,True,True,False,True,57.525216,85.451775,103.010467,0.978292


In [3]:
print(MLBAM18.columns.tolist())

['inning', 'batterId', 'pitcherId', 'event', 'x', 'y', 'ab_num', 'timestamp', 'stand', 'throws', 'runnerMovement', 'half', 'balls', 'strikes', 'endOuts', 'actionId', 'description', 'game_type', 'home_team', 'home_teamId', 'home_lg', 'away_team', 'away_teamId', 'away_lg', 'venueId', 'stadium', 'field_teamId', 'playerId.1B', 'playerId.2B', 'playerId.3B', 'playerId.C', 'playerId.CF', 'playerId.LF', 'playerId.RF', 'playerId.SS', 'batterPos', 'batterName', 'pitcherName', 'runsOnPlay', 'startOuts', 'runsInInning', 'runsITD', 'runsFuture', 'start1B', 'start2B', 'start3B', 'end1B', 'end2B', 'end3B', 'outsInInning', 'startCode', 'endCode', 'fielderId', 'gameId', 'isPA', 'isAB', 'isHit', 'isBIP', 'our.x', 'our.y', 'r', 'theta']


In [4]:
columns_we_need = ['batterName', 'batterId', 'event',\
                  'start1B', 'start2B', 'start3B', \
                  'end1B', 'end2B', 'end3B', \
                  'startOuts', 'endOuts', 'runsFuture', \
                  'runsOnPlay', 'outsInInning', \
                  'stand', 'throws', 'venueId', 'stadium', 
                  'batterPos']
RE18 = MLBAM18[columns_we_need].copy()
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF


In [5]:
RE18['Start1'] = np.where(pd.isnull(RE18.start1B), 0, 1)
RE18['Start2'] = np.where(pd.isnull(RE18.start2B), 0, 1)
RE18['Start3'] = np.where(pd.isnull(RE18.start3B), 0, 1)
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0


In [6]:
RE18['Start_State'] = (RE18['Start1'].astype(str) + \
                       RE18['Start2'].astype(str) + \
                       RE18['Start3'].astype(str) + ' ' + \
                       RE18['startOuts'].astype(str))
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1


In [7]:
RE18['End1'] = np.where(pd.isnull(RE18.end1B),0,1)
RE18['End2'] = np.where(pd.isnull(RE18.end2B),0,1)
RE18['End3'] = np.where(pd.isnull(RE18.end3B),0,1)
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1


In [8]:

RE18['End_State'] = (RE18.End1.astype(str) + \
                     RE18.End2.astype(str) + \
                     RE18.End3.astype(str) + ' ' +
                     RE18.endOuts.astype(str))

RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0,000 0
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0,100 0
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0,110 0
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0,110 1
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1,011 2


In [9]:
RE18 = RE18[((RE18.Start_State != RE18.End_State) | \
             (RE18.runsOnPlay > 0)) & \
             (RE18.outsInInning == 3)]
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0,000 0
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0,100 0
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0,110 0
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0,110 1
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1,011 2


In [10]:
# Calculate runs in remainder of inning for each state. 
Start_RunExp = (RE18.groupby(['Start_State'])['runsFuture']
                    .mean()
                    .reset_index()
                    .rename(columns={'runsFuture':'Start_RE'}))
Start_RunExp.head()

,Start_State,Start_RE
0,000 0,0.492774
1,000 1,0.264412
2,000 2,0.097473
3,001 0,1.424437
4,001 1,1.017470


In [11]:
# Merge run expectancy into MLBAM data
# based on original state

RE18 = pd.merge(RE18, Start_RunExp,
                on=['Start_State'],
                how='left')

RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0,000 0,0.492774
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0,100 0,0.492774
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0,110 0,0.869314
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0,110 1,1.420023
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1,011 2,0.933985


In [12]:
# Create series of observations for base state when there are 3 outs.

Base_State_3 = [pd.Series(['000 3', 0], index=Start_RunExp.columns),
                pd.Series(['001 3', 0], index=Start_RunExp.columns),
                pd.Series(['010 3', 0], index=Start_RunExp.columns),
                pd.Series(['011 3', 0], index=Start_RunExp.columns),
                pd.Series(['100 3', 0], index=Start_RunExp.columns),
                pd.Series(['101 3', 0], index=Start_RunExp.columns),
                pd.Series(['110 3', 0], index=Start_RunExp.columns),
                pd.Series(['111 3', 0], index=Start_RunExp.columns)]

Base_State_3

[Start_State    000 3
 Start_RE           0
 dtype: object,
 Start_State    001 3
 Start_RE           0
 dtype: object,
 Start_State    010 3
 Start_RE           0
 dtype: object,
 Start_State    011 3
 Start_RE           0
 dtype: object,
 Start_State    100 3
 Start_RE           0
 dtype: object,
 Start_State    101 3
 Start_RE           0
 dtype: object,
 Start_State    110 3
 Start_RE           0
 dtype: object,
 Start_State    111 3
 Start_RE           0
 dtype: object]

In [13]:
# Add these states to Start_RunExp, 
# then rename this df as End_RunExp so we can
# merge the values back into RE18.

Start_RunExp = Start_RunExp.append(Base_State_3, 
                                   ignore_index=True)
col_dict = {'Start_State': 'End_State',
            'Start_RE': 'End_RE'} 
End_RunExp = Start_RunExp.rename(columns=col_dict)
End_RunExp.head()

,End_State,End_RE
0,000 0,0.492774
1,000 1,0.264412
2,000 2,0.097473
3,001 0,1.424437
4,001 1,1.017470


In [14]:
# Merge run expectanancy int RE18 based on End_State
RE18 = pd.merge(RE18, End_RunExp, on=['End_State'], how='left')
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0,000 0,0.492774,0.492774
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0,100 0,0.492774,0.869314
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0,110 0,0.869314,1.420023
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0,110 1,1.420023,0.933985
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1,011 2,0.933985,0.611722


In [15]:
# Calculate Run Value of each event

RE18['Run_Value'] = RE18.runsOnPlay + \
                    RE18.End_RE - \
                    RE18.Start_RE
RE18.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE,Run_Value
0,"Happ, I",664023,Home Run,NaN,NaN,NaN,NaN,NaN,NaN,0,0,3,1,3,L,R,4169,Marlins Park,CF,0,0,0,000 0,0,0,0,000 0,0.492774,0.492774,1.000000
1,Bryant,592178,Walk,NaN,NaN,NaN,592178.0,NaN,NaN,0,0,2,0,3,R,R,4169,Marlins Park,3B,0,0,0,000 0,1,0,0,100 0,0.492774,0.869314,0.376540
2,Rizzo,519203,Hit By Pitch,592178.0,NaN,NaN,519203.0,592178.0,NaN,0,0,2,0,3,L,R,4169,Marlins Park,1B,1,0,0,100 0,1,1,0,110 0,0.869314,1.420023,0.550709
3,Contreras,575929,Strikeout,519203.0,592178.0,NaN,519203.0,592178.0,NaN,0,1,2,0,3,R,R,4169,Marlins Park,C,1,1,0,110 0,1,1,0,110 1,1.420023,0.933985,-0.486038
4,Schwarber,656941,Groundout,519203.0,592178.0,NaN,NaN,519203.0,592178.0,1,2,2,0,3,L,R,4169,Marlins Park,LF,1,1,0,110 1,0,1,1,011 2,0.933985,0.611722,-0.322263


In [16]:
Event_Value = (RE18.groupby(['event'])['Run_Value']
               .mean()
               .reset_index())
Event_Value.sort_values(by='Run_Value',
                       ascending=False).head()

,event,Run_Value
16,Home Run,1.379867
28,Triple,1.111033
5,Double,0.769674
9,Fielders Choice,0.686499
7,Fan interference,0.615308


In [17]:
Player_Value = (RE18.groupby(['batterName'])['Run_Value']
                .sum()
                .reset_index())
Player_Value.sort_values(by='Run_Value',
                         ascending=False).head()

,batterName,Run_Value
554,"Martinez, J",75.124792
80,Betts,65.724469
907,Trout,62.286430
980,Yelich,60.583786
715,"Ramirez, J",53.113122


# I. Coding Run Expectancy Dataset (2017)

In [18]:
# Read in MLBAM Data for 2017

MLBAM17 = pd.read_csv("MLBAM17.csv")

In [19]:
MLBAM17.drop(['Unnamed: 0'], axis=1, inplace=True)

MLBAM17.head()

,inning,batterId,pitcherId,event,x,y,ab_num,timestamp,stand,throws,runnerMovement,half,balls,strikes,endOuts,actionId,description,game_type,home_team,home_teamId,home_lg,away_team,away_teamId,away_lg,venueId,stadium,field_teamId,playerId.1B,playerId.2B,playerId.3B,playerId.C,playerId.CF,playerId.LF,playerId.RF,playerId.SS,batterPos,batterName,pitcherName,runsOnPlay,startOuts,runsInInning,runsITD,runsFuture,start1B,start2B,start3B,end1B,end2B,end3B,outsInInning,startCode,endCode,fielderId,gameId,isPA,isAB,isHit,isBIP,our.x,our.y,r,theta
0,1,458731,502042,Flyout,65.59,123.50,1,2017-04-02 17:12:18,L,R,NaN,top,2,2,1,NaN,Brett Gardner flies out to left fielder Mallex...,R,tba,139,AL,nya,147,AL,12,Tropicana Field,139,489149,543543,446334,519083,595281,605480,519306,542921,LF,Gardner,Archer,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,3,0,0,605480.0,gid_2017_04_02_nyamlb_tbamlb_1,True,True,False,True,-148.267814,188.423160,239.763700,2.237490
1,1,596142,502042,Groundout,128.19,193.88,2,2017-04-02 17:15:42,R,R,NaN,top,0,0,2,NaN,"Gary Sanchez grounds out sharply, pitcher Chri...",R,tba,139,AL,nya,147,AL,12,Tropicana Field,139,489149,543543,446334,519083,595281,605480,519306,542921,C,"Sanchez, G",Archer,0,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,3,0,0,502042.0,gid_2017_04_02_nyamlb_tbamlb_1,True,True,False,True,7.961190,12.777835,15.055020,1.013603
2,1,595885,502042,Walk,NaN,NaN,3,2017-04-02 17:16:48,L,R,[595885::1B::Walk],top,4,1,2,NaN,Greg Bird walks.,R,tba,139,AL,nya,147,AL,12,Tropicana Field,139,489149,543543,446334,519083,595281,605480,519306,542921,1B,Bird,Archer,0,2,0,0,0,NaN,NaN,NaN,595885.0,NaN,NaN,3,0,1,NaN,gid_2017_04_02_nyamlb_tbamlb_1,True,False,False,False,NaN,NaN,NaN,NaN
3,1,407812,502042,Groundout,153.48,157.29,4,2017-04-02 17:19:09,R,R,[595885:1B:::Groundout],top,3,1,3,NaN,"Yankees challenged (play at 1st), call on the ...",R,tba,139,AL,nya,147,AL,12,Tropicana Field,139,489149,543543,446334,519083,595281,605480,519306,542921,DH,Holliday,Archer,0,2,0,0,0,595885.0,NaN,NaN,NaN,NaN,NaN,3,1,0,543543.0,gid_2017_04_02_nyamlb_tbamlb_1,True,True,False,True,71.076710,104.094437,126.045827,0.971701
4,1,572816,547888,Single,140.75,73.31,5,2017-04-02 17:24:43,L,R,[572816::1B::Single],bottom,0,1,0,NaN,Corey Dickerson singles on a sharp line drive ...,R,tba,139,AL,nya,147,AL,12,Tropicana Field,147,595885,516770,452104,596142,453056,458731,592450,591720,DH,"Dickerson, C",Tanaka,0,0,3,0,3,NaN,NaN,NaN,572816.0,NaN,NaN,3,0,1,NaN,gid_2017_04_02_nyamlb_tbamlb_1,True,True,True,True,39.306818,313.680887,316.134030,1.446138


In [20]:
RE17 = MLBAM17[columns_we_need]
RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH


In [21]:
import warnings
warnings.filterwarnings('ignore')

In [22]:
RE17['Start1'] = np.where(pd.isnull(RE17['start1B']),0,1)
RE17['Start2'] = np.where(pd.isnull(RE17['start2B']),0,1)
RE17['Start3'] = np.where(pd.isnull(RE17['start3B']),0,1)

RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0


In [23]:
RE17['Start_State'] = (RE17['Start1'].astype(str) + \
                       RE17['Start2'].astype(str) + \
                       RE17['Start3'].astype(str) + \
                       " " + RE17['startOuts'].astype(str))

RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0


In [24]:
RE17['End1'] = np.where(pd.isnull(RE17['end1B']),0,1)
RE17['End2'] = np.where(pd.isnull(RE17['end2B']),0,1)
RE17['End3'] = np.where(pd.isnull(RE17['end3B']),0,1)
RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0


In [25]:
RE17['End_State'] = (RE17['End1'].astype(str) + \
                     RE17['End2'].astype(str) + \
                     RE17['End3'].astype(str) + \
                     ' ' + RE17['endOuts'].astype(str))

RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0,000 1
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0,000 2
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0,100 2
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0,000 3
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0,100 0


In [26]:
RE17 = RE17[((RE17.Start_State != RE17.End_State) |
             (RE17.runsOnPlay > 0)) & (RE17.outsInInning == 3)]

RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0,000 1
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0,000 2
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0,100 2
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0,000 3
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0,100 0


In [27]:
# Calculate runs in remainder of inning for each state.

Start_RunExp = (RE17.groupby(['Start_State'])['runsFuture']
                .mean()
                .reset_index()
                .rename(columns={'runsFuture':'Start_RE'}))
Start_RunExp.head()

,Start_State,Start_RE
0,000 0,0.516375
1,000 1,0.272176
2,000 2,0.108038
3,001 0,1.436482
4,001 1,0.953586


In [28]:
RE17 = pd.merge(RE17, Start_RunExp,
                on=['Start_State'],
                how='left')
RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0,000 1,0.516375
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0,000 2,0.272176
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0,100 2,0.108038
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0,000 3,0.225365
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0,100 0,0.516375


In [29]:
# Create series of observations for base state when there are 3 outs.

Base_State_3 = [pd.Series(['000 3', 0], index=Start_RunExp.columns),
                pd.Series(['001 3', 0], index=Start_RunExp.columns),
                pd.Series(['010 3', 0], index=Start_RunExp.columns),
                pd.Series(['011 3', 0], index=Start_RunExp.columns),
                pd.Series(['100 3', 0], index=Start_RunExp.columns),
                pd.Series(['101 3', 0], index=Start_RunExp.columns),
                pd.Series(['110 3', 0], index=Start_RunExp.columns),
                pd.Series(['111 3', 0], index=Start_RunExp.columns)]

Base_State_3

[Start_State    000 3
 Start_RE           0
 dtype: object,
 Start_State    001 3
 Start_RE           0
 dtype: object,
 Start_State    010 3
 Start_RE           0
 dtype: object,
 Start_State    011 3
 Start_RE           0
 dtype: object,
 Start_State    100 3
 Start_RE           0
 dtype: object,
 Start_State    101 3
 Start_RE           0
 dtype: object,
 Start_State    110 3
 Start_RE           0
 dtype: object,
 Start_State    111 3
 Start_RE           0
 dtype: object]

In [32]:
# And these states to Start_RunExp
# then rename this df as End_RunExp so we can
# merge the values back into RE18
Start_RunExp = Start_RunExp.append(Base_State_3, ignore_index=True)

End_RunExp = Start_RunExp.rename(columns={'Start_State':'End_State',
                                          'Start_RE': 'End_RE'})

End_RunExp.head()

,End_State,End_RE
0,000 0,0.516375
1,000 1,0.272176
2,000 2,0.108038
3,001 0,1.436482
4,001 1,0.953586


In [33]:
# Merge run expectancy into RE18 based on End_State
RE17 = pd.merge(RE17, End_RunExp, on=['End_State'], how='left')

RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0,000 1,0.516375,0.272176
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0,000 2,0.272176,0.108038
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0,100 2,0.108038,0.225365
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0,000 3,0.225365,0.000000
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0,100 0,0.516375,0.912921


In [34]:
RE17['Run_Value'] = RE17.runsOnPlay + RE17.End_RE - RE17.Start_RE
RE17.head()

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos,Start1,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE,Run_Value
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF,0,0,0,000 0,0,0,0,000 1,0.516375,0.272176,-0.244199
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C,0,0,0,000 1,0,0,0,000 2,0.272176,0.108038,-0.164138
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B,0,0,0,000 2,1,0,0,100 2,0.108038,0.225365,0.117327
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH,1,0,0,100 2,0,0,0,000 3,0.225365,0.000000,-0.225365
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH,0,0,0,000 0,1,0,0,100 0,0.516375,0.912921,0.396546


In [35]:
Event_Value = RE17.groupby(['event'])['Run_Value'].mean().reset_index()
Event_Value.sort_values('Run_Value', ascending=False).head()

,event,Run_Value
16,Home Run,1.378739
27,Triple,1.063607
5,Double,0.779338
9,Fielders Choice,0.764112
7,Fan interference,0.590743


In [37]:
Player_Value = (RE17.groupby(['batterName'])['Run_Value']
                .sum()
                .reset_index())

Player_Value.sort_values(by=['Run_Value'],
                         ascending=False).head()

,batterName,Run_Value
909,Votto,67.650742
87,Blackmon,62.032793
445,Judge,57.081625
37,Arenado,56.111576
331,Goldschmidt,56.078941


In [40]:
# Calculate percent of plate appearances resulting in flyouts

RE17['Count'] = 1
Flyout = RE17[RE17['event']=='Flyout']
sum(Flyout['Count'])/sum(RE17['Count'])

0.10697100421648598

In [ ]:
# How many 

In [30]:
#Your Code Here


# II. Coding Run Expectancy Dataset (2016)

In [31]:
# Read in MLBAM Data for 2016

MLBAM16 = pd.read_csv("../Data/MLBAM16.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../Data/MLBAM16.csv'

In [ ]:
#Your Code Here

# III. Comparing 2016 vs. 2017

In [ ]:
#Your Code Here